In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

In [2]:
us = pd.read_csv('Cleaned US Electric Vehicle.csv')

In [3]:
us

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Base MSRP,Legislative District,Vehicle Location,Electric Utility,2020 Census Tract
0,3C3CFFGE4E,Yakima,Yakima,WA,98902.0,2014,FIAT,500,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,87,0,14.0,POINT (-120.524012 46.5973939),PACIFICORP,5.307700e+10
1,5YJXCBE40H,Thurston,Olympia,WA,98513.0,2017,TESLA,MODEL X,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,200,0,2.0,POINT (-122.817545 46.98876),PUGET SOUND ENERGY INC,5.306701e+10
2,3MW39FS03P,King,Renton,WA,98058.0,2023,BMW,330E,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,20,0,11.0,POINT (-122.1298876 47.4451257),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
3,7PDSGABA8P,Snohomish,Bothell,WA,98012.0,2023,RIVIAN,R1S,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0,0,21.0,POINT (-122.1873 47.820245),PUGET SOUND ENERGY INC,5.306105e+10
4,5YJ3E1EB8L,King,Kent,WA,98031.0,2020,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,322,0,33.0,POINT (-122.2012521 47.3931814),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10305,WP0CE2A78N,King,Vashon,WA,98070.0,2022,PORSCHE,PANAMERA,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,18,0,34.0,POINT (-122.46049 47.44873),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
10306,YV4BR0CK5K,Pierce,Puyallup,WA,98372.0,2019,VOLVO,XC90,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,17,0,31.0,POINT (-122.28718 47.190465),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.305394e+10
10307,WP0CD2Y10N,Pierce,Gig Harbor,WA,98335.0,2022,PORSCHE,TAYCAN,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0,0,26.0,POINT (-122.5835454 47.3234488),BONNEVILLE POWER ADMINISTRATION||CITY OF TACOM...,5.305307e+10
10308,YV4ER3XM7R,King,Seattle,WA,98122.0,2024,VOLVO,XC40,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0,0,43.0,POINT (-122.30839 47.610365),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10


In [4]:
us.columns

Index(['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year',
       'Make', 'Model', 'Electric Vehicle Type',
       'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range',
       'Base MSRP', 'Legislative District', 'Vehicle Location',
       'Electric Utility', '2020 Census Tract'],
      dtype='object')

In [5]:
us['Electric Utility'].value_counts()

Electric Utility
PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA)                                                                       4147
CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA)                                                                        2614
PUGET SOUND ENERGY INC                                                                                              1625
BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF CLARK COUNTY - (WA)                                                     949
BONNEVILLE POWER ADMINISTRATION||CITY OF TACOMA - (WA)||PENINSULA LIGHT COMPANY                                      190
BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF COWLITZ COUNTY                                                           93
PUGET SOUND ENERGY INC||PUD NO 1 OF WHATCOM COUNTY                                                                    87
PACIFICORP                                                                                                            79
BONNEVILLE POWE

In [11]:
us.drop(['VIN (1-10)', 'Postal Code','Vehicle Location','2020 Census Tract','Electric Utility'], axis = 1 , inplace = True)

In [13]:
us.columns

Index(['County', 'City', 'State', 'Model Year', 'Make', 'Model',
       'Electric Vehicle Type',
       'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range',
       'Base MSRP', 'Legislative District'],
      dtype='object')

In [15]:
def encode_cafv(eligibility):
    if 'Clean Alternative Fuel Vehicle Eligible' in eligibility:
        return 2
    elif 'Not eligible' in eligibility:
        return 0
    else:
        return 1  # For "Eligibility unknown"
        
us['CAFV_Encoded'] = us['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].apply(encode_cafv)


In [17]:
categorical_cols = ['County', 'City', 'State', 'Make', 'Model', 
                    'Electric Vehicle Type']
us = pd.get_dummies(us, columns=categorical_cols, drop_first=True)


In [19]:
us.columns

Index(['Model Year', 'Clean Alternative Fuel Vehicle (CAFV) Eligibility',
       'Electric Range', 'Base MSRP', 'Legislative District', 'CAFV_Encoded',
       'County_Benton', 'County_Chelan', 'County_Clallam', 'County_Clark',
       ...
       'Model_VOLT', 'Model_WHEEGO', 'Model_WRANGLER', 'Model_X3', 'Model_X5',
       'Model_XC40', 'Model_XC60', 'Model_XC90', 'Model_XM',
       'Electric Vehicle Type_Plug-in Hybrid Electric Vehicle (PHEV)'],
      dtype='object', length=460)

In [21]:
## GOal - Estimate the expected electric range based on features like vehicle type, model, and price.



In [23]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [24]:
# Replace long strings like 'Eligibility unknown...' with NaN and convert to numeric
us['Electric Range'] = pd.to_numeric(us['Electric Range'], errors='coerce')

# Drop rows where Electric Range is still NaN
us = us.dropna(subset=['Electric Range'])

In [27]:
# Convert Electric Range to numeric (in case non-numeric values exist)
us['Electric Range'] = pd.to_numeric(us['Electric Range'], errors='coerce')

# Drop rows with missing Electric Range
us.dropna(subset=['Electric Range'], inplace=True)


In [30]:
X = us.drop(columns=['Electric Range'])
y = us['Electric Range']

In [32]:
 #Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
# Train model
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor()
model.fit(X_train, y_train)

ValueError: could not convert string to float: 'Eligibility unknown as battery range has not been researched'

In [38]:
y_pred = model.predict(X_test)


ValueError: could not convert string to float: 'Clean Alternative Fuel Vehicle Eligible'

In [ ]:
y_pred

In [ ]:
print("R² Score:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))


In [ ]:
## check for overfitting 

In [ ]:
print("Train R²:", model.score(X_train, y_train))
print("Test R²:", model.score(X_test, y_test))

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual Electric Range")
plt.ylabel("Predicted Electric Range")
plt.title("Actual vs Predicted Electric Range")
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.nlargest(15).plot(kind='barh')
plt.title("Top 15 Feature Importances")
plt.show()


## Predict if a vehicle is CAVF Elligble or not 

In [ ]:
us = us.dropna(subset=['CAFV_Encoded'])


In [ ]:
# Only keep rows with 0 and 2
us = us[us['CAFV_Encoded'].isin([0, 2])].copy()


In [ ]:
X = us.drop(columns=['CAFV_Encoded'])
y = us['CAFV_Encoded'].map({0: 0, 2: 1})  # Convert to binary: 0 = Not Eligible, 1 = Eligible


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.ensemble import RandomForestClassifier


In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)


In [ ]:
us['CAFV_Encoded'].value_counts()

In [ ]:
y_pred = model.predict(X_test)


In [ ]:
y_pred

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# feature importance 
pd.Series(model.feature_importances_, index=X.columns).nlargest(15).plot(kind='barh')


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Only for numeric features
corr = us.corr(numeric_only=True)
plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Feature Correlation")
plt.show()


In [ ]:
## Estimate MSRP Based on vehicle type and specs 

In [ ]:
us_price = us.drop(columns=[
    'Base MSRP',
    'CAFV_Encoded',
])
